# 🎙️ Générateur de voix Mindset 24

Modèle **Chatterbox Multilingual** (Resemble AI, licence MIT : usage commercial autorisé), français, clonage de voix à partir d'un extrait de 10 à 30 s.

**Avant de commencer :** menu *Exécution → Modifier le type d'exécution → GPU T4*. Puis lance les cellules dans l'ordre (▶).

⚠️ Ne clone qu'une voix dont la licence autorise le clonage, ou la voix d'une personne qui a donné son accord.

In [ ]:
#@title 1. Installation (2–3 min, une fois par session)
!pip -q install chatterbox-tts pydub
import torch; print('GPU :', torch.cuda.is_available())

In [ ]:
#@title 2. Chargement du modèle (1–2 min)
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChatterboxMultilingualTTS.from_pretrained(device=DEV)
print('prêt', model.sr, 'Hz')

In [ ]:
#@title 3. Voix de référence (extrait propre de 10–30 s, sans musique)
#@markdown Laisse vide pour envoyer un fichier depuis ton téléphone/PC.
VOIX_URL = '' #@param {type:'string'}
NOM_VOIX = 'voix_mindset24' #@param {type:'string'}
import os, subprocess
os.makedirs('voix', exist_ok=True)
if VOIX_URL:
    subprocess.run(['wget', '-q', '-O', 'voix/src', VOIX_URL], check=True); src = 'voix/src'
else:
    from google.colab import files
    up = files.upload(); src = list(up.keys())[0]
REF = f'voix/{NOM_VOIX}.wav'
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src, '-ac', '1', '-ar', '24000', '-t', '30', REF], check=True)
print('référence :', REF)

## 4. Texte
Une phrase ou un passage par ligne. Au début d'une ligne, tu peux mettre une émotion :
- `[calme]` voix posée · `[intense]` voix tendue · `[cri]` voix criée · rien = normal
- une ligne vide = un silence de 0,6 s

In [ ]:
#@title 4. Texte à dire
TEXTE = '''
[calme] Personne ne croyait en lui.
Kaï, épisode un.

[intense] Alors montre-leur qui tu es !
[cri] MONTRE-LEUR !
'''
VITESSE = 1.0 #@param {type:'number'}

In [ ]:
#@title 5. Générer la voix
import re, numpy as np
EMO = {'calme': (0.35, 0.6), 'intense': (0.75, 0.4), 'cri': (1.1, 0.3), '': (0.5, 0.5)}  # (exagération, cfg)
morceaux = []
sil = lambda s: np.zeros(int(model.sr * s), dtype=np.float32)
for ligne in TEXTE.strip().split('\n'):
    ligne = ligne.strip()
    if not ligne: morceaux.append(sil(0.6)); continue
    m = re.match(r'\[(\w+)\]\s*(.*)', ligne); emo, txt = (m.group(1), m.group(2)) if m else ('', ligne)
    ex, cfg = EMO.get(emo, EMO[''])
    wav = model.generate(txt, language_id='fr', audio_prompt_path=REF, exaggeration=ex, cfg_weight=cfg)
    morceaux += [wav.squeeze(0).cpu().numpy(), sil(0.25)]
    print('✓', emo or 'normal', '|', txt)
audio = np.concatenate(morceaux)
ta.save('voix_brute.wav', torch.from_numpy(audio).unsqueeze(0), model.sr)
!ffmpeg -v error -y -i voix_brute.wav -af "atempo={VITESSE},loudnorm=I=-14" -b:a 192k voix.mp3
from IPython.display import Audio; Audio('voix.mp3')

In [ ]:
#@title 6. Récupérer le fichier (téléchargement + copie dans Google Drive)
from google.colab import files, drive
import shutil, datetime
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/Mindset24/voix', exist_ok=True)
nom = f"voix_{datetime.date.today():%Y%m%d}.mp3"
shutil.copy('voix.mp3', f'/content/drive/MyDrive/Mindset24/voix/{nom}')
files.download('voix.mp3'); print('copié dans Drive : Mindset24/voix/' + nom)